In [ ]:
!pip install -q --upgrade ultralytics          # YOLO 11 포함 (v8.3+)
!pip install -q scikit-learn pandas tqdm

In [ ]:
import os, yaml, gc, pathlib, pprint, uuid, shutil
from glob import glob
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from ultralytics import YOLO
import numpy as np
import cv2
import skimage.measure as sm
import os
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import tarfile
dataset_tar_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Dataset/Yolo_splited_train_test_266.tar.gz'
data_path = '/content'
with tarfile.open(dataset_tar_path, mode="r:*") as tar:  # r:* = 압축 방식 자동 판별
    tar.extractall(path=data_path)

## Test data loading

In [ ]:
DATA_ROOT = pathlib.Path('/content/dataset/YOLO_Test_266')          # 사용자 지시
IMG_EXT = ('.jpg')

images, labels = [], []
for img_path in DATA_ROOT.rglob('*'):
    if img_path.suffix.lower() in IMG_EXT:
        lbl_path = pathlib.Path(str(img_path).replace('/images/', '/labels/')).with_suffix('.txt')
        if lbl_path.exists():
            images.append(str(img_path))
            labels.append(str(lbl_path))
print(f'✓ 유효 페어: {len(images):,}')
print(images, labels)

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def plot_annotations(image_path, label_path, class_names, ax):
    """Reads YOLO format annotations and plots them on the image."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img.shape

    ax.imshow(img)
    ax.set_title(os.path.basename(image_path))
    ax.axis('off')

    with open(label_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            class_id = int(parts[0])
            class_name = class_names[class_id]

            if len(parts) > 5:  # Segmentation annotations
                # Convert normalized polygon points to absolute pixel coordinates
                polygon_points = np.array(parts[1:], dtype=float).reshape(-1, 2)
                polygon_points[:, 0] *= width
                polygon_points[:, 1] *= height

                # Create a mask from the polygon
                mask = np.zeros((height, width), dtype=np.uint8)
                cv2.fillPoly(mask, [np.int32(polygon_points)], color=1)

                # Apply the mask to a colored overlay for visualization
                color = plt.cm.get_cmap('tab10')(class_id % 10)[:3] # Get a color based on class_id
                colored_mask = np.zeros_like(img)
                colored_mask[mask == 1] = (np.array(color) * 255).astype(np.uint8)
                ax.imshow(colored_mask, alpha=0.2) # Adjust alpha for transparency

                # Add text label for segmentation
                # Find the centroid of the polygon to place the text
                M = cv2.moments(np.int32(polygon_points))
                if M["m00"] != 0:
                    cX = int(M["m10"] / M["m00"])
                    cY = int(M["m01"] / M["m00"])
                    ax.text(cX, cY, class_name, color='white', fontsize=8, bbox=dict(facecolor=plt.cm.get_cmap('tab10')(class_id % 10), alpha=0.5))


            else: # Bounding box annotations (optional, if your labels also contain boxes)
                # Convert normalized bounding box coordinates to absolute pixel coordinates
                x_center, y_center, bbox_width, bbox_height = map(float, parts[1:])
                x_min = (x_center - bbox_width / 2) * width
                y_min = (y_center - bbox_height / 2) * height
                rect = patches.Rectangle((x_min, y_min), bbox_width * width, bbox_height * height,
                                         linewidth=1, edgecolor=plt.cm.get_cmap('tab10')(class_id % 10), facecolor='none')
                ax.add_patch(rect)
                ax.text(x_min, y_min, class_name, color='white', fontsize=8, bbox=dict(facecolor=plt.cm.get_cmap('tab10')(class_id % 10), alpha=0.5))


# Define class names
class_names = ['Tooth', 'Root']

# Randomly select 20 image and label pairs
num_samples_to_plot = 20
if len(images) >= num_samples_to_plot:
    sampled_indices = random.sample(range(len(images)), num_samples_to_plot)
else:
    sampled_indices = range(len(images)) # Plot all if less than 20

selected_images = [images[i] for i in sampled_indices]
selected_labels = [labels[i] for i in sampled_indices]

# Visualize the selected images and annotations
fig, axes = plt.subplots(5, 4, figsize=(20, 25))
axes = axes.flatten()

for i, (img_path, lbl_path) in enumerate(zip(selected_images, selected_labels)):
    plot_annotations(img_path, lbl_path, class_names, axes[i])

plt.tight_layout()
plt.show()

In [ ]:
def get_classes_in_label(lbl):
    classes = set()
    with open(lbl) as f:
        for line in f:
            items = line.split()
            if items:
                classes.add(int(items[0]))
    # Return a sorted list of unique class IDs as a string for stratification
    return ','.join(map(str, sorted(list(classes))))

# Modify y to contain a string representation of all classes in each label file
y = [get_classes_in_label(l) for l in labels]
print(y)

sss = StratifiedShuffleSplit(n_splits=5, test_size=0.20, random_state=42)
folds = list(sss.split(images, y))

## Model training

In [ ]:
import os
import numpy as np

# Paths to the ground truth labels folders
# Updated to use only the specified folder
labels_folders = [
    '/content/dataset/YOLO_train_266/labels/train'
]

# Dictionary to store instance counts per image
image_instance_counts = {}

# Define class IDs for 'Tooth' and 'Root' based on your dataset's data.yaml (assuming 0 for Tooth, 1 for Root)
# You should verify these IDs from your actual data.yaml file
class_id_tooth = 0
class_id_root = 1

# Iterate through each specified labels folder
for labels_folder in labels_folders:
    print(f"Attempting to read label files from: {labels_folder}") # Added debug print
    if os.path.exists(labels_folder):
        print(f"Labels folder found at: {labels_folder}") # Added debug print
        for label_file in os.listdir(labels_folder):
            if label_file.endswith('.txt'):
                image_id = os.path.splitext(label_file)[0]
                label_path = os.path.join(labels_folder, label_file)

                # Initialize counts for the current image if not already present
                if image_id not in image_instance_counts:
                    image_instance_counts[image_id] = {'Tooth': 0, 'Root': 0}

                try:
                    with open(label_path, 'r') as f:
                        for line in f:
                            parts = line.strip().split()
                            if len(parts) > 0:
                                class_id = int(parts[0])
                                if class_id == class_id_tooth:
                                    image_instance_counts[image_id]['Tooth'] += 1
                                elif class_id == class_id_root:
                                    image_instance_counts[image_id]['Root'] += 1
                except Exception as e:
                    print(f"Error reading file {label_file}: {e}")
    else:
        print(f"Warning: Labels folder not found at {labels_folder}. Skipping this folder.") # Added debug print


print(f"Finished processing label files. Processed annotations for {len(image_instance_counts)} unique images.") # Added debug print
print(f"Keys in image_instance_counts (first 5): {list(image_instance_counts.keys())[:5]}") # Added debug print to show some keys


# Calculate average counts and standard deviations
total_images_with_annotations = len(image_instance_counts)

# Explicitly print the number of analyzed images
print(f"\nTotal images analyzed for statistics: {total_images_with_annotations}") # Added debug print


if total_images_with_annotations > 0:
    # Extract counts into lists for calculation across all collected images
    tooth_counts = [counts['Tooth'] for counts in image_instance_counts.values()]
    root_counts = [counts['Root'] for counts in image_instance_counts.values()]

    total_tooth_instances = sum(tooth_counts)
    total_root_instances = sum(root_counts)

    average_tooth_count = total_tooth_instances / total_images_with_annotations
    average_root_count = total_root_instances / total_images_with_annotations

    std_tooth_count = np.std(tooth_counts) if tooth_counts else 0
    std_root_count = np.std(root_counts) if root_counts else 0


    print(f"\nAverage number of 'Tooth' instances per image: {average_tooth_count:.2f}")
    print(f"Standard deviation of 'Tooth' instances per image: {std_tooth_count:.2f}")
    print(f"Average number of 'Root' instances per image: {average_root_count:.2f}")
    print(f"Standard deviation of 'Root' instances per image: {std_root_count:.2f}")
else:
    print("No images with annotations found in the specified folders to calculate averages and standard deviations.")

In [ ]:
!pip install -q "faster-coco-eval>=1.6.7"

In [ ]:
hyperparams = {
    "lr0": 0.001,  # Lower learning rate for medical images
    "lrf": 0.01,  # Learning rate final fraction
    "momentum": 0.937,  # Momentum for optimizer
    "weight_decay": 0.0005,  # L2 regularization
    "warmup_epochs": 3.0,  # Number of warmup epochs
    "batch_size": 8,  # Adjust based on GPU memory
    "epochs": 70,  # Total training epochs
    "patience": 10,  # Early stopping patience
    "imgsz": 768  # Ensure image size is 640
}

# Define data augmentation settings
augmentations = {
    "degrees": 15,  # Rotation ±15 degrees
    "scale": 0.1,  # Scale factor (0.9~1.1)
    "shear": 2.0,  # Shearing augmentation
    "flipud": 0.5,  # Vertical flipping probability
    "fliplr": 0.5,  # Horizontal flipping probability
 #   "hsv_h": 0.015,  # Hue augmentation
 #   "hsv_s": 0.7,  # Saturation augmentation
    "hsv_v": 0.4  # Brightness augmentation
}

In [ ]:
# ───────────────── 3. Fold 루프 ─────────────────────────────
results = []

# ─── 3. Fold 학습 루프 ───────────────────────────────────────────────────────
for k, (tr, va) in enumerate(folds, 1):
    print(f'\n🚀 Fold {k}/5 — train {len(tr)} | val {len(va)}')

    tr_imgs = [images[i] for i in tr]
    va_imgs = [images[i] for i in va]

    # 3‑1. 이미지 경로 리스트 → .txt
    tr_txt, va_txt = f'/content/train{k}.txt', f'/content/val{k}.txt'
    with open(tr_txt, 'w') as f: f.write('\n'.join(tr_imgs))
    with open(va_txt, 'w') as f: f.write('\n'.join(va_imgs))

    # 3‑2. data.yaml 생성
    yaml_path = f'/content/data_fold{k}_{uuid.uuid4().hex[:6]}.yaml'
    with open(yaml_path, 'w') as f:
        yaml.safe_dump({
            'path': '.',
            'train': tr_txt,
            'val':   va_txt,
            'nc': 2,
            'names': ['Tooth', 'Root']
        }, f, sort_keys=False)

    # 3‑3. ✨ Segmentation 모델 로드 (Large) ✨
    model = YOLO('yolo11m-seg.pt', task='segment')

    # 3‑4. 🔧 하이퍼파라미터 강화 학습
    model.train(
        data=yaml_path,
        epochs=hyperparams["epochs"],
        imgsz=hyperparams["imgsz"],  # Ensure image size is set to 640
        batch=hyperparams["batch_size"],
        lr0=hyperparams["lr0"],
        lrf=hyperparams["lrf"],
        momentum=hyperparams["momentum"],
        weight_decay=hyperparams["weight_decay"],
        warmup_epochs=hyperparams["warmup_epochs"],
        augment=True,  # Enable data augmentation
        degrees=augmentations["degrees"],
        scale=augmentations["scale"],
        shear=augmentations["shear"],
        flipud=augmentations["flipud"],
        fliplr=augmentations["fliplr"],
 #       hsv_h=augmentations["hsv_h"],
 #       hsv_s=augmentations["hsv_s"],
         hsv_v=augmentations["hsv_v"],
        patience=hyperparams["patience"],
        project='/content/yolo1m_seg_toothroot',
        name=f'fold{k}',
        exist_ok=True,
        cache=False,
        device = 0
    )

    # 3‑5. 🔍 향상된 검증 설정 (IoU 0.65, 큰 스케일)
    metrics = model.val(data=yaml_path)

    # Print the entire metrics object to inspect its structure
    print("\nDebugging: Full metrics object structure")
    pprint.pprint(metrics)


    # 3‑6. 메트릭 수집 (seg 네임스페이스) ---------------------------
    results.append({
      'fold': k,
      "Total_mAP50":     metrics.seg.map50,   # ← 메서드
      "Total_mAP50‑95": metrics.seg.map,     # ← 메서드
      "Total_Precision": metrics.seg.mp,      # ← 수정
      "Total_Recall":    metrics.seg.mr,      # ← 수정
      "Total_F1":        float(np.mean(metrics.seg.f1)),
      "Tooth_mAP50": metrics.seg.ap50[0],   # mAP50 for class 0 (Tooth) - Corrected
      "Tooth_mAP50-95": metrics.seg.ap[0],  # mAP50-95 for class 0 (Tooth) - Corrected
      "Tooth_Precision": metrics.seg.p[0],  # Precision for class 0 (Tooth) - Corrected
      "Tooth_Recall": metrics.seg.r[0],     # Recall for class 0 (Tooth) - Corrected
      "Tooth_F1":        float(metrics.seg.f1[0]), # Corrected
      "Root_mAP50": metrics.seg.ap50[1],    # mAP50 for class 1 (Root) - Corrected
      "Root_mAP50-95": metrics.seg.ap[1],   # mAP50-95 for class 1 (Root) - Corrected
      "Root_Precision": metrics.seg.p[1],   # Precision for class 1 (Root) - Corrected
      "Root_Recall": metrics.seg.r[1],      # Recall for class 1 (Root) - Corrected
      "Root_F1":        float(metrics.seg.f1[1]), # Corrected
    })

    del model; gc.collect()
    # 파일은 남겨 두어도 됨 (디버깅용)

In [ ]:
# prompt: Results 변수 내용을 파일로 저장해 줘

# Save results to a file
results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/Colab Notebooks/Root_seg/YOLO_0726_Result/training_results_0712.csv', index=False)

print("Results saved to training_results.csv")

In [ ]:
# prompt: 'Results' dataframe의 각 변수에 대해 평균과 95%Confidence interval를 계산하고 파일로 저장해 줘

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), scipy.stats.sem(a)
    h = se * scipy.stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h

import scipy.stats

# Calculate mean and confidence interval for each metric
summary_results = {}
for col in results_df.columns:
    if col != 'fold': # Skip the 'fold' column
        mean, ci_lower, ci_upper = mean_confidence_interval(results_df[col])
        summary_results[col] = {
            'Mean': mean,
            'CI_Lower': ci_lower,
            'CI_Upper': ci_upper
        }

# Convert the summary results to a DataFrame for easier viewing and saving
summary_df = pd.DataFrame(summary_results).T

# Save the summary results to a file
summary_file_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/YOLO_0725_Result/training_results_95CI.csv'
summary_df.to_csv(summary_file_path)
print(summary_df)
print(f"Summary results (Mean and 95% CI) saved to {summary_file_path}")
print("\nSummary Results:")
summary_df


In [ ]:
cp "/content/fold4_backup.tar.gz" "/content/drive/MyDrive/Colab Notebooks/Root_seg/YOLO_0725_Result"

In [ ]:
import tarfile
import os

# Path to the folder you want to compress
source_dir = '/content/yolo1m_seg_toothroot/fold4'

# Output tar file path
tar_output_path = '/content/fold4_backup.tar.gz'

# Create the tar.gz archive
with tarfile.open(tar_output_path, 'w:gz') as tar:
    tar.add(source_dir, arcname=os.path.basename(source_dir))

print(f"Compressed {source_dir} into {tar_output_path}")

In [ ]:
%pip install ultralytics

In [ ]:
from ultralytics import YOLO

# Load the pre-trained model
model_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/YOLO_0725_Result_best/best_4_best.pt'
model = YOLO(model_path)

print(f"Model loaded successfully from {model_path}")

In [ ]:

from ultralytics.utils.plotting import colors
# 원하는 색을 BGR 로 지정 (len ≥ 클래스 개수)
colors.palette[0] = (135, 206, 250)   # Tooth (class 0) → light‑blue (#ADD8E6)
colors.palette[1] = ( 255, 165, 0)   # Root  (class 1) → orange   (#FFA500)

In [ ]:
import cv2
import os
from pathlib import Path
import numpy as np
from PIL import Image # Import Pillow for mask creation
from collections import defaultdict # Import defaultdict to group masks by class

image_folder = '/content/dataset/YOLO_Test_266/images' # Assuming this is the folder used for prediction source

# Assuming 'model' is already loaded from a previous cell

# Define the output directory for combined per-class masks
save_dir_combined_per_class = Path("/content/runs/mask_Yolo_combined_per_class") # New output folder
save_dir_combined_per_class.mkdir(parents=True, exist_ok=True)
print(f"Saving combined predicted masks per class to: {save_dir_combined_per_class}")


# Run prediction and store results as a list for multiple access
print("Running model prediction...")
# Ensure the model variable exists
if 'model' not in locals():
    print("Error: 'model' variable not found. Please load the model first.")
    results_list = [] # Initialize as empty if model not found
else:
    # Run prediction with stream=True and immediately convert the generator to a list
    prediction_results_generator = model.predict(
        source= image_folder,   # Image folder
        stream=True,             # Return a generator
        save=False,              # Turn off default saving by model
        retina_masks=True,       # Get masks in original resolution
        show_boxes=False,        # Hide boxes
        show_labels=False,       # Hide labels
        show_conf=False          # Hide confidence scores
    )
    results_list = list(prediction_results_generator) # Convert generator to a list


if not results_list:
    print("No prediction results were generated.")
else:
    print(f"Generated predictions for {len(results_list)} images.")
    # Process results and save combined masks per class
    print("Processing prediction results and saving combined masks per class...")
    for i, r in enumerate(results_list): # Iterate over the list
        image_path = r.path
        image_id = Path(image_path).stem # Get image stem (filename without extension)

        if r.masks is not None and hasattr(r.masks, 'data') and hasattr(r.boxes, 'cls'):
            masks_tensor = r.masks.data # Get the mask tensor (N, H, W)
            class_ids = r.boxes.cls # Get class IDs for each mask

            if masks_tensor.shape[0] > 0: # Check if there are any masks predicted for this image
                # Get the shape of the masks (assuming all masks for an image have the same shape)
                mask_h, mask_w = masks_tensor.shape[1:]

                # Group masks by class ID
                masks_by_class_id = defaultdict(list)
                for j in range(masks_tensor.shape[0]):
                     # Ensure mask tensor is on CPU before converting to numpy
                     mask_np = masks_tensor[j].cpu().numpy().astype(np.uint8) # Convert to uint8 (0 or 1)
                     class_id = int(class_ids[j])
                     masks_by_class_id[class_id].append(mask_np)


                # Process and save combined mask for each class
                for class_id, mask_list_for_class in masks_by_class_id.items():
                    if mask_list_for_class: # Ensure there are masks for this class
                        # Create a blank black image with the same dimensions as the masks for this class
                        combined_class_mask_np = np.zeros((mask_h, mask_w), dtype=np.uint8) # Start with a black background

                        # Combine all instance masks for this class
                        for mask_np in mask_list_for_class:
                            # Use bitwise OR to combine masks (any pixel that is 1 in any mask becomes 1 in the combined mask)
                             if mask_np.shape == combined_class_mask_np.shape:
                                combined_class_mask_np = np.logical_or(combined_class_mask_np, mask_np).astype(np.uint8)
                             else:
                                # This should ideally not happen if masks are from the same prediction run
                                print(f"Warning: Shape mismatch for a mask in class {class_id} for image {image_id}. Skipping combination for this mask instance.")


                        # Create a PIL Image from the combined class mask numpy array (mode 'L' for grayscale 0-255)
                        # Multiply by 255 to make the mask visible (white on black background)
                        combined_class_mask_img = Image.fromarray(combined_class_mask_np * 255, mode='L')

                        # Get class name (assuming model.names is available)
                        class_name = model.names.get(class_id, f"class_{class_id}")

                        # Define the output filename for the combined class mask
                        # Example: image_id_combined_mask_Tooth.png
                        combined_class_mask_filename = f"{image_id}_combined_mask_{class_name}.png"
                        combined_class_mask_output_path = save_dir_combined_per_class / combined_class_mask_filename

                        # Save the combined class mask image
                        try:
                            combined_class_mask_img.save(combined_class_mask_output_path)
                            print(f"Saved combined mask for class '{class_name}' for image: {image_id}")
                        except Exception as e:
                            print(f"Error saving combined mask for class '{class_name}' for image {image_id}: {e}")

            else:
                print(f"No masks predicted for image: {image_id}. No combined masks per class saved.")

        else:
            print(f"No masks found in results for image: {image_id}")

    print("\nFinished saving combined predicted masks per class.")

In [ ]:


import os
import subprocess

source_dir = '/content/runs/mask_custom'
output_tar = '/content/mask_custom_black.tar' # Name of the output tar file

print(f"Checking if source directory exists: {source_dir}")
# Check if the source directory exists
if not os.path.isdir(source_dir):
    print(f"Error: Source directory not found: {source_dir}")
else:
    print(f"Source directory found. Creating tar archive: {output_tar}")
    # Construct the tar command
    # -c: create an archive
    # -v: verbosely list files processed
    # -f: use archive file or device (specify output_tar)
    # Specify the directory to archive
    tar_command = ['tar', '-cvf', output_tar, '-C', os.path.dirname(source_dir), os.path.basename(source_dir)]

    try:
        # Execute the command
        subprocess.run(tar_command, check=True, capture_output=True, text=True)
        print(f"Successfully created tar archive: {output_tar}")
        # Optional: Print stdout/stderr for debugging
        # print("STDOUT:", result.stdout)
        # print("STDERR:", result.stderr)

    except subprocess.CalledProcessError as e:
        print(f"Error creating tar archive: {e}")
        print(f"Command output (stderr):\n{e.stderr}")
    except FileNotFoundError:
        print("Error: 'tar' command not found. Make sure tar is installed.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

In [ ]:
import os

# Path to the ground truth labels folder
labels_folder = '/content/dataset/YOLO_Test_266/labels/train'

# Dictionary to store ground truth annotations
ground_truth = {}

# Iterate through each label file in the folder
for label_file in os.listdir(labels_folder):
    if label_file.endswith('.txt'):
        image_id = os.path.splitext(label_file)[0]
        label_path = os.path.join(labels_folder, label_file)

        # Read the annotations from the file
        annotations = []
        with open(label_path, 'r') as f:
            for line in f:
                # Assuming YOLO format: class_id center_x center_y width height
                parts = line.strip().split()
                if len(parts) == 5:
                    class_id = int(parts[0])
                    center_x, center_y, width, height = map(float, parts[1:])
                    annotations.append({
                        'class_id': class_id,
                        'center_x': center_x,
                        'center_y': center_y,
                        'width': width,
                        'height': height
                    })
        ground_truth[image_id] = annotations

print(f"Loaded annotations for {len(ground_truth)} images.")

## Modify pixel metrics calculation for bootstrapping

### Subtask:
Update the `calculate_pixel_counts` function in cell `dc3951e2` to return pixel counts (TP, TN, FP, FN) per image for each class.


**Reasoning**:
I need to modify the `calculate_pixel_counts` function to return pixel counts per image instead of aggregated totals. I will update the function in the specified cell to achieve this.



In [ ]:
from collections import defaultdict
import pandas as pd
import os
import numpy as np
from PIL import Image, ImageDraw
import scipy.stats
from sklearn.utils import resample # Using sklearn for bootstrapping

def calculate_iou_segmentation(mask1, mask2):
    """
    Calculates the Intersection over Union (IoU) of two segmentation masks.
    Masks are expected to be boolean numpy arrays of the same shape.
    """
    # Ensure masks have the same shape before performing operations
    if mask1.shape != mask2.shape:
        # This should ideally not happen if masks are generated correctly based on image size
        # Handle or report the error appropriately
        # print(f"Shape mismatch in IoU calculation: {mask1.shape} vs {mask2.shape}")
        return 0.0 # Return 0 IoU if shapes don't match

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    iou = intersection / (union + 1e-6) # Add epsilon to avoid division by zero
    return iou


def evaluate_predictions_segmentation_for_bootstrap(ground_truth_masks, predictions, iou_threshold=0.5):
    """
    Evaluates predictions against ground truth segmentation masks for bootstrapping.
    Returns counts of TP, FP, FN per image (instance-based).
    """
    image_tp_fp_fn = {}

    for image_id in ground_truth_masks.keys(): # Iterate through all images in GT
        gt_class_masks = ground_truth_masks.get(image_id, {})
        predicted_annotations = predictions.get(image_id, [])

        current_image_tp = defaultdict(int)
        current_image_fp = defaultdict(int)
        current_image_fn = defaultdict(int)

        # Keep track of matched ground truth masks for this image
        matched_ground_truth_masks = {class_id: [False] * len(masks) for class_id, masks in gt_class_masks.items()}

        for pred in predicted_annotations:
            pred_class_id = pred['class_id']
            pred_mask = pred['mask']

            if pred_class_id not in gt_class_masks:
                current_image_fp[pred_class_id] += 1
                continue

            gt_masks_for_class = gt_class_masks.get(pred_class_id, [])
            if not gt_masks_for_class:
                 current_image_fp[pred_class_id] += 1
                 continue

            best_iou = 0
            best_gt_match_index = -1

            for i, gt_mask in enumerate(gt_masks_for_class):
                if not matched_ground_truth_masks[pred_class_id][i]:
                    if pred_mask.shape != gt_mask.shape:
                         iou = 0.0
                    else:
                        iou = calculate_iou_segmentation(pred_mask, gt_mask)

                    if iou > best_iou:
                        best_iou = iou
                        best_gt_match_index = i

            if best_iou >= iou_threshold:
                current_image_tp[pred_class_id] += 1
                matched_ground_truth_masks[pred_class_id][best_gt_match_index] = True
            else:
                current_image_fp[pred_class_id] += 1

        # Count false negatives for this image
        for class_id, gt_masks_for_class in gt_class_masks.items():
            for i, gt_mask in enumerate(gt_masks_for_class):
                if not matched_ground_truth_masks[class_id][i]:
                    current_image_fn[class_id] += 1

        image_tp_fp_fn[image_id] = {
            'tp': current_image_tp,
            'fp': current_image_fp,
            'fn': current_image_fn
        }

    return image_tp_fp_fn


def bootstrap_metrics(image_tp_fp_fn, num_bootstrap_samples=1000, confidence=0.95, class_names=None):
    """
    Performs bootstrapping to calculate confidence intervals for Precision, Recall, and F1-score (instance-based).
    image_tp_fp_fn is a dictionary {image_id: {'tp': {class_id: count}, 'fp': {class_id: count}, 'fn': {class_id: count}}}
    """
    image_ids = list(image_tp_fp_fn.keys())
    n_images = len(image_ids)
    class_ids = sorted(list(set([cid for img_data in image_tp_fp_fn.values() for metric_type in ['tp', 'fp', 'fn'] for cid in img_data.get(metric_type, {}).keys()])))

    bootstrap_results = defaultdict(lambda: defaultdict(list)) # {class_id: {metric: [values]}}

    for _ in range(num_bootstrap_samples):
        # Resample image IDs with replacement
        resampled_image_ids = resample(image_ids, n_samples=n_images, replace=True)

        # Aggregate TP, FP, FN for the resampled images
        bootstrap_tp = defaultdict(int)
        bootstrap_fp = defaultdict(int)
        bootstrap_fn = defaultdict(int)

        for img_id in resampled_image_ids:
            img_data = image_tp_fp_fn[img_id]
            for class_id in class_ids:
                bootstrap_tp[class_id] += img_data['tp'].get(class_id, 0)
                bootstrap_fp[class_id] += img_data['fp'].get(class_id, 0)
                bootstrap_fn[class_id] += img_data['fn'].get(class_id, 0)

        # Calculate metrics for the resampled data
        for class_id in class_ids:
            tp = bootstrap_tp[class_id]
            fp = bootstrap_fp[class_id]
            fn = bootstrap_fn[class_id]

            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
            f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

            bootstrap_results[class_id]['Precision'].append(precision)
            bootstrap_results[class_id]['Recall'].append(recall)
            bootstrap_results[class_id]['F1-score'].append(f1)

        # Calculate overall metrics for the resampled data (micro-average)
        total_tp = sum(bootstrap_tp.values())
        total_fp = sum(bootstrap_fp.values())
        total_fn = sum(bootstrap_fn.values())

        overall_precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
        overall_recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
        overall_f1 = 2 * (overall_precision * overall_recall) / (overall_precision + overall_recall) if (overall_precision + overall_recall) > 0 else 0.0

        bootstrap_results['Overall']['Precision'].append(overall_precision)
        bootstrap_results['Overall']['Recall'].append(overall_recall)
        bootstrap_results['Overall']['F1-score'].append(overall_f1)


    # Calculate mean and confidence intervals from bootstrap results
    metric_summary = {}
    alpha = 1.0 - confidence
    for class_id, metrics in bootstrap_results.items():
        class_name = class_names.get(class_id, f'Class {class_id}') if class_id != 'Overall' else 'Overall'
        metric_summary[class_name] = {}
        for metric_name, values in metrics.items():
            mean = np.mean(values)
            # Calculate confidence interval using percentiles
            lower_bound = np.percentile(values, alpha / 2.0 * 100)
            upper_bound = np.percentile(values, (1.0 - alpha / 2.0) * 100)
            metric_summary[class_name][metric_name] = {
                'Mean': mean,
                'CI_Lower': lower_bound,
                'CI_Upper': upper_bound
            }

    return metric_summary


def load_ground_truth_segmentation_masks(labels_folder, image_folder, target_size=(768, 448)):
    """
    Loads ground truth segmentation masks from YOLO segmentation annotation files (.txt).
    Parses polygon coordinates, draws masks, and resizes them to a target size.
    """
    gt_masks = {}
    print(f"Loading ground truth segmentation masks from {labels_folder} and resizing to {target_size}...")

    for label_file in os.listdir(labels_folder):
        if label_file.endswith('.txt'):
            image_id = os.path.splitext(os.path.basename(label_file))[0]
            label_path = os.path.join(labels_folder, label_file)
            image_path = os.path.join(image_folder, image_id + '.jpg') # Assuming images are .jpg

            if not os.path.exists(image_path):
                print(f"Warning: Image file not found for {image_id}. Skipping annotations for this image.")
                continue

            # Get original image size to scale normalized coordinates
            try:
                with Image.open(image_path) as img:
                    original_img_w, original_img_h = img.size
            except Exception as e:
                print(f"Error opening image {image_id}: {e}. Skipping annotations.")
                continue

            gt_masks[image_id] = defaultdict(list)

            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 3: # Need at least class_id and one point (x, y)
                        continue

                    class_id = int(parts[0])
                    # The rest are normalized polygon coordinates (x1 y1 x2 y2 ... xn yn)
                    polygon_coords_norm = list(map(float, parts[1:]))

                    # Ensure there are an even number of polygon coordinates (x, y pairs)
                    if len(polygon_coords_norm) % 2 != 0:
                        print(f"Warning: Odd number of polygon coordinates in {label_file}. Skipping annotation.")
                        continue

                    # Convert normalized coordinates to pixel coordinates based on original image size
                    polygon_coords_original_pixels = []
                    for i in range(0, len(polygon_coords_norm), 2):
                        x = int(polygon_coords_norm[i] * original_img_w)
                        y = int(polygon_coords_norm[i+1] * original_img_h)
                        polygon_coords_original_pixels.extend([x, y])

                    # Create a blank mask image with the original image dimensions
                    mask_img_original = Image.new('L', (original_img_w, original_img_h), 0) # 'L' mode for grayscale (0 or 255)
                    draw = ImageDraw.Draw(mask_img_original)

                    # Draw the polygon on the mask
                    draw.polygon(polygon_coords_original_pixels, fill=255) # Fill the polygon with white (255)

                    # Resize the mask to the target size (matching predicted mask size)
                    mask_img_resized = mask_img_original.resize(target_size, resample=Image.NEAREST) # Use NEAREST for mask resizing

                    # Convert the resized mask image to a boolean numpy array
                    mask_np_resized = np.array(mask_img_resized) > 0 # Convert to boolean True/False


                    if mask_np_resized.sum() > 0: # Only add if the mask is not empty after resizing
                         gt_masks[image_id][class_id].append(mask_np_resized)
                    # else:
                        # print(f"Warning: Generated empty mask after resizing for an annotation in {label_file}. Skipping.")


    print(f"Loaded and resized masks for {len(gt_masks)} images.")
    return gt_masks


# Function to calculate pixel-wise TP, TN, FP, FN per image for Accuracy and Specificity
def calculate_pixel_counts_per_image(ground_truth_masks, predictions, class_names=None):
    """
    Calculates pixel-wise True Positives, True Negatives, False Positives, and False Negatives
    for each class, for each image.
    Assumes ground_truth_masks and predictions contain masks of the same dimensions.
    Returns a dictionary {image_id: {'tp': {class_id: count}, 'tn': {class_id: count}, 'fp': {class_id: count}, 'fn': {class_id: count}}}
    """
    image_pixel_counts = {}

    # Get all unique class IDs from ground truth and predictions
    all_class_ids = set()
    for image_id, class_masks in ground_truth_masks.items():
        all_class_ids.update(class_masks.keys())
    for image_id, predicted_annotations in predictions.items():
        all_class_ids.update([p['class_id'] for p in predicted_annotations])
    all_class_ids = sorted(list(all_class_ids))


    print("Calculating pixel counts per image per class...")

    for image_id in ground_truth_masks.keys(): # Iterate through images present in GT
        gt_class_masks = ground_truth_masks.get(image_id, {})
        predicted_annotations = predictions.get(image_id, [])

        current_image_pixel_counts = {
            'tp': defaultdict(int),
            'tn': defaultdict(int),
            'fp': defaultdict(int),
            'fn': defaultdict(int)
        }

        # Get image dimensions from ground truth masks (assuming masks are consistent)
        img_h, img_w = 0, 0
        for class_id, masks in gt_class_masks.items():
             if masks:
                img_h, img_w = masks[0].shape
                break

        if img_h == 0:
             # If no GT masks for this image, try to get dimensions from predicted masks
            if predicted_annotations and predicted_annotations[0].get('mask') is not None:
                img_h, img_w = predicted_annotations[0]['mask'].shape
                # print(f"Warning: Using predicted mask shape ({img_h}, {img_w}) for image {image_id} as no GT masks found.")
            else:
                 print(f"Warning: Could not determine image size for image {image_id}. Skipping pixel counts.")
                 continue # Cannot calculate pixel counts without dimensions


        total_pixels_in_image = img_h * img_w

        # Calculate pixel counts for each class in this image
        for class_id in all_class_ids:
            # Create a single ground truth mask for this class in this image (union of all instances)
            gt_mask_for_class = np.zeros((img_h, img_w), dtype=bool)
            if class_id in gt_class_masks:
                 for mask in gt_class_masks[class_id]:
                     if mask.shape == gt_mask_for_class.shape:
                        gt_mask_for_class = np.logical_or(gt_mask_for_class, mask)
                     else:
                        # print(f"Shape mismatch for class GT mask {class_id} in image {image_id}: {mask.shape} vs expected {gt_mask_for_class.shape}. Skipping.")
                        pass # Skip if shape mismatch, continue with empty mask for this instance


            # Create a single predicted mask for this class in this image (union of all instances)
            pred_mask_for_class = np.zeros((img_h, img_w), dtype=bool)
            # Find predictions for this class in this image
            preds_for_class = [p['mask'] for p in predicted_annotations if p['class_id'] == class_id]
            for pred_mask in preds_for_class:
                 if pred_mask.shape == pred_mask_for_class.shape:
                    pred_mask_for_class = np.logical_or(pred_mask_for_class, pred_mask)
                 else:
                    # print(f"Shape mismatch for class Pred mask {class_id} in image {image_id}: {pred_mask.shape} vs expected {pred_mask_for_class.shape}. Skipping.")
                    pass # Skip if shape mismatch, continue with empty mask for this instance


            # Ensure shapes match before pixel-wise calculation
            if gt_mask_for_class.shape != pred_mask_for_class.shape:
                 print(f"Shape mismatch between class {class_id} GT ({gt_mask_for_class.shape}) and Pred ({pred_mask_for_class.shape}) masks for image {image_id}. Skipping pixel counts for this class in this image.")
                 continue


            # Calculate pixel counts for this class in this image
            tp_class = np.logical_and(gt_mask_for_class, pred_mask_for_class).sum()
            fp_class = np.logical_and(~gt_mask_for_class, pred_mask_for_class).sum() # GT is background for this class, Pred is foreground
            fn_class = np.logical_and(gt_mask_for_class, ~pred_mask_for_class).sum() # GT is foreground for this class, Pred is background
            # TN for a specific class: pixels that are background in both GT and prediction *for this class*.
            # This is equivalent to total pixels minus TP, FP, and FN for this class.
            tn_class = total_pixels_in_image - tp_class - fp_class - fn_class


            # Store counts for this image and class
            current_image_pixel_counts['tp'][class_id] = tp_class
            current_image_pixel_counts['tn'][class_id] = tn_class
            current_image_pixel_counts['fp'][class_id] = fp_class
            current_image_pixel_counts['fn'][class_id] = fn_class

        image_pixel_counts[image_id] = current_image_pixel_counts

    return image_pixel_counts


def bootstrap_pixel_metrics(image_pixel_counts, num_bootstrap_samples=1000, confidence=0.95, class_names=None):
    """
    Performs bootstrapping to calculate confidence intervals for pixel-based Accuracy and Specificity.
    image_pixel_counts is a dictionary {image_id: {'tp': {class_id: count}, 'tn': {class_id: count}, 'fp': {class_id: count}, 'fn': {class_id: count}}}
    """
    image_ids = list(image_pixel_counts.keys())
    n_images = len(image_ids)

    # Determine all unique class IDs present in the pixel counts
    all_class_ids = set()
    for img_data in image_pixel_counts.values():
        for metric_type in ['tp', 'tn', 'fp', 'fn']:
            all_class_ids.update(img_data.get(metric_type, {}).keys())
    all_class_ids = sorted(list(all_class_ids))


    bootstrap_results = defaultdict(lambda: defaultdict(list)) # {class_id: {metric: [values]}}

    print(f"Performing bootstrapping for pixel metrics with {num_bootstrap_samples} samples...")


    for _ in range(num_bootstrap_samples):
        # Resample image IDs with replacement
        resampled_image_ids = resample(image_ids, n_samples=n_images, replace=True)

        # Aggregate pixel counts for the resampled images
        bootstrap_tp_pixels = defaultdict(int)
        bootstrap_tn_pixels = defaultdict(int)
        bootstrap_fp_pixels = defaultdict(int)
        bootstrap_fn_pixels = defaultdict(int)

        for img_id in resampled_image_ids:
            img_data = image_pixel_counts[img_id]
            for class_id in all_class_ids:
                 bootstrap_tp_pixels[class_id] += img_data['tp'].get(class_id, 0)
                 bootstrap_tn_pixels[class_id] += img_data['tn'].get(class_id, 0)
                 bootstrap_fp_pixels[class_id] += img_data['fp'].get(class_id, 0)
                 bootstrap_fn_pixels[class_id] += img_data['fn'].get(class_id, 0)


        # Calculate pixel metrics for the resampled data
        for class_id in all_class_ids:
            tp = bootstrap_tp_pixels[class_id]
            tn = bootstrap_tn_pixels[class_id]
            fp = bootstrap_fp_pixels[class_id]
            fn = bootstrap_fn_pixels[class_id]

            total_pixels = tp + tn + fp + fn
            accuracy = (tp + tn) / total_pixels if total_pixels > 0 else 0.0
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

            class_name = class_names.get(class_id, f'Class {class_id}')
            bootstrap_results[class_name]['Accuracy'].append(accuracy)
            bootstrap_results[class_name]['Specificity'].append(specificity)

        # Calculate overall pixel metrics (across all classes combined) for the resampled data
        total_tp_overall = sum(bootstrap_tp_pixels.values())
        total_tn_overall = sum(bootstrap_tn_pixels.values())
        total_fp_overall = sum(bootstrap_fp_pixels.values())
        total_fn_overall = sum(bootstrap_fn_pixels.values())

        total_pixels_overall = total_tp_overall + total_tn_overall + total_fp_overall + total_fn_overall
        overall_accuracy = (total_tp_overall + total_tn_overall) / total_pixels_overall if total_pixels_overall > 0 else 0.0
        overall_specificity = total_tn_overall / (total_tn_overall + total_fp_overall) if (total_tn_overall + total_fp_overall) > 0 else 0.0

        bootstrap_results['Overall']['Accuracy'].append(overall_accuracy)
        bootstrap_results['Overall']['Specificity'].append(overall_specificity)


    # Calculate mean and confidence intervals from bootstrap results
    metric_summary = {}
    alpha = 1.0 - confidence

    for class_name, metrics in bootstrap_results.items():
        metric_summary[class_name] = {}
        for metric_name, values in metrics.items():
            mean = np.mean(values)
            lower_bound = np.percentile(values, alpha / 2.0 * 100)
            upper_bound = np.percentile(values, (1.0 - alpha / 2.0) * 100)
            metric_summary[class_name][metric_name] = {
                'Mean': mean,
                'CI_Lower': lower_bound,
                'CI_Upper': upper_bound
            }

    return metric_summary


# Assuming 'results' from model.predict() is available and 'ground_truth_segmentation_masks' is loaded

# Example of how to process results from model.predict() for segmentation
# The 'results' object from model.predict() is a list of Results objects
predictions = {}
# Add checks for the structure of 'results'
if isinstance(results, list) and all(hasattr(r, 'masks') for r in results):
    for result in results:
        image_path = result.path
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        predictions[image_id] = []
        if result.masks is not None:
            # result.masks is a Masks object
            # result.masks.data contains the boolean masks as a tensor (N, H, W)
            # result.boxes contains bounding box info, including class_id and confidence
            if hasattr(result.masks, 'data') and hasattr(result.boxes, 'cls') and hasattr(result.boxes, 'conf'):
                masks_tensor = result.masks.data # Use .data for the tensor
                class_ids = result.boxes.cls
                scores = result.boxes.conf

                # Predicted masks are likely already at the processing size (e.g., 448x768)
                # We will use their shape as the target size for resizing GT masks.
                # Assuming all predicted masks for an image have the same shape.
                if masks_tensor.shape[0] > 0:
                    predicted_mask_shape = masks_tensor.shape[1:] # (H, W)
                    target_size = (predicted_mask_shape[1], predicted_mask_shape[0]) # (W, H)
                else:
                    # Fallback if no masks are predicted for this image
                    target_size = (768, 448) # Default target size, should match expected prediction size


                for i in range(masks_tensor.shape[0]):
                     # Ensure mask tensor is on CPU before converting to numpy
                     pred_mask_np = masks_tensor[i].cpu().numpy().astype(bool)

                     predictions[image_id].append({
                        'class_id': int(class_ids[i]),
                        'mask': pred_mask_np, # Use the predicted mask as is
                        'score': float(scores[i])
                    })
            else:
                 print(f"Warning: Unexpected mask or box format in result for image {image_id}")
else:
    print("Error: 'results' variable is not in the expected format for processing segmentation predictions.")
    predictions = {} # Reset predictions to empty if format is wrong


# Determine the target size for resizing ground truth masks based on predicted mask shape
target_size_for_gt = (768, 448) # Default, will try to get from predictions if available
if predictions:
    # Find the shape of the first predicted mask in the first image with predictions
    for image_id, predicted_annotations in predictions.items():
        if predicted_annotations and predicted_annotations[0].get('mask') is not None:
            predicted_mask_shape = predicted_annotations[0]['mask'].shape # (H, W)
            target_size_for_gt = (predicted_mask_shape[1], predicted_mask_shape[0]) # (W, H)
            print(f"Determined target size for GT masks from predictions: {target_size_for_gt}")
            break # Found a predicted mask shape, no need to check further

# Load ground truth segmentation masks, resizing them to match predicted mask size
labels_folder = '/content/dataset/YOLO_Test_266/labels/train'
image_folder = '/content/dataset/YOLO_Test_266/images/train' # Need image folder to get image sizes
ground_truth_segmentation_masks = load_ground_truth_segmentation_masks(labels_folder, image_folder, target_size=target_size_for_gt)


# Evaluate predictions and get instance-based TP, FP, FN per image for bootstrapping (PRF)
image_tp_fp_fn_instance = evaluate_predictions_segmentation_for_bootstrap(
    ground_truth_segmentation_masks, predictions, iou_threshold=0.5
)

# Calculate instance-based metrics and 95% Confidence Intervals using bootstrapping (PRF)
metric_summary_with_ci = bootstrap_metrics(
    image_tp_fp_fn_instance,
    num_bootstrap_samples=1000, # Number of bootstrap samples
    confidence=0.95,
    class_names=model.names
)

# Convert instance-based metric summary with CI to DataFrame for display
metrics_ci_df = pd.DataFrame({
    (outerKey, innerKey): innerValue for outerKey, outerDict in metric_summary_with_ci.items() for innerKey, innerValue in outerDict.items()
}).T.swaplevel(0, 1).sort_index()


print("\nInstance-based Performance Metrics with 95% Confidence Intervals (Segmentation):")
display(metrics_ci_df)


# Calculate pixel-wise TP, TN, FP, FN per image for Accuracy and Specificity
image_pixel_counts_per_image_result = calculate_pixel_counts_per_image(ground_truth_segmentation_masks, predictions, class_names=model.names)

# Calculate pixel-based metrics (Accuracy, Specificity) with 95% Confidence Intervals using bootstrapping
pixel_metrics_ci_summary = bootstrap_pixel_metrics(
    image_pixel_counts_per_image_result,
    num_bootstrap_samples=1000, # Number of bootstrap samples
    confidence=0.95,
    class_names=model.names
)

print("\nPixel-based Performance Metrics with 95% Confidence Intervals (Segmentation):")

# Convert pixel metrics with CI to DataFrame for display
pixel_metrics_ci_df = pd.DataFrame({
    (outerKey, innerKey): innerValue for outerKey, outerDict in pixel_metrics_ci_summary.items() for innerKey, innerValue in outerDict.items()
}).T.swaplevel(0, 1).sort_index()

display(pixel_metrics_ci_df)


# You can also save these DataFrames to files if needed
metrics_ci_df.to_csv('/content/instance_metrics_segmentation_with_ci.csv')
pixel_metrics_ci_df.to_csv('/content/pixel_metrics_segmentation_with_ci.csv')
# print("\nPerformance metrics saved to files.")

In [ ]:
import os
import shutil

# Define source and destination directories
source_dir = '/content'
destination_dir = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare'

print(f"Checking if destination directory exists: {destination_dir}")
# Create the destination directory if it doesn't exist
if not os.path.exists(destination_dir):
    print(f"Destination directory does not exist. Creating: {destination_dir}")
    try:
        os.makedirs(destination_dir)
        print(f"Successfully created directory: {destination_dir}")
    except OSError as e:
        print(f"Error creating directory {destination_dir}: {e}")
        # Stop execution if directory cannot be created
        raise

print(f"Looking for CSV files in source directory: {source_dir}")
# List all files in the source directory
files_in_source = os.listdir(source_dir)

csv_files_to_copy = [f for f in files_in_source if f.endswith('.csv')]

if not csv_files_to_copy:
    print(f"No CSV files found in {source_dir} to copy.")
else:
    print(f"Found {len(csv_files_to_copy)} CSV files to copy: {csv_files_to_copy}")
    print(f"Copying files to {destination_dir}...")

    # Copy each CSV file to the destination directory
    for csv_file in csv_files_to_copy:
        source_path = os.path.join(source_dir, csv_file)
        destination_path = os.path.join(destination_dir, csv_file)

        try:
            shutil.copy2(source_path, destination_path) # copy2 attempts to preserve metadata
            print(f"Successfully copied: {csv_file} to {destination_dir}")
        except IOError as e:
            print(f"Error copying file {csv_file}: {e}")
        except Exception as e:
            print(f"An unexpected error occurred while copying {csv_file}: {e}")

    print("Finished attempting to copy CSV files.")

In [ ]:
# Calculate mAP50 for segmentation with 95% Confidence Interval (Class-wise and Overall)

from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.utils import resample # Using sklearn for bootstrapping

def calculate_iou_segmentation(mask1, mask2):
    """
    Calculates the Intersection over Union (IoU) of two segmentation masks.
    Masks are expected to be boolean numpy arrays of the same shape.
    """
    # Ensure masks have the same shape before performing operations
    if mask1.shape != mask2.shape:
        # This should ideally not happen if masks are generated correctly based on image size
        # Handle or report the error appropriately
        # print(f"Shape mismatch in IoU calculation: {mask1.shape} vs {mask2.shape}")
        return 0.0 # Return 0 IoU if shapes don't match

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    iou = intersection / (union + 1e-6) # Add epsilon to avoid division by zero
    return iou

def calculate_ap(true_positives, false_positives, total_ground_truth):
    """
    Calculates Average Precision (AP) from lists of true positives and false positives
    at different confidence thresholds, and the total number of ground truth objects.
    """
    # Calculate cumulative true positives and false positives
    cum_tp = np.cumsum(true_positives)
    cum_fp = np.cumsum(false_positives)

    # Calculate precision and recall
    precision = cum_tp / (cum_tp + cum_fp + 1e-6)
    recall = cum_tp / (total_ground_truth + 1e-6)

    # Append (0, 1) and (1, 0) points to the precision-recall curve
    precision = np.concatenate(([1.], precision, [0.]))
    recall = np.concatenate(([0.], recall, [1.]))

    # Interpolate the precision-recall curve by taking the maximum precision
    # for each recall level down the curve
    for i in range(len(precision) - 2, -1, -1):
        precision[i] = np.maximum(precision[i], precision[i + 1])

    # Calculate AP as the area under the interpolated precision-recall curve using all points
    ap = np.sum((recall[:-1] != recall[1:]) * precision[:-1] * (recall[1:] - recall[:-1]))

    return ap


def calculate_mAP_segmentation_for_bootstrap(image_predictions, ground_truth_masks, iou_threshold=0.5):
    """
    Calculates AP for each class and overall mAP for a given set of image predictions and ground truth masks.
    Used internally for bootstrapping. Returns a dictionary of APs {class_id: ap_value}.
    """
    class_aps = {}

    # Sort predictions by confidence score in descending order
    image_predictions.sort(key=lambda x: x['score'], reverse=True)

    # Get unique class IDs from predictions
    predicted_class_ids = list(set([pred['class_id'] for pred in image_predictions]))

    # Get unique class IDs from ground truth masks
    gt_class_ids = set()
    for image_id, class_masks in ground_truth_masks.items():
        gt_class_ids.update(class_masks.keys())

    # Combine unique class IDs from both predictions and ground truth
    unique_class_ids = sorted(list(set(predicted_class_ids + list(gt_class_ids))))


    for class_id in unique_class_ids:
        # Filter predictions and ground truth for the current class
        class_predictions = [pred for pred in image_predictions if pred['class_id'] == class_id]
        class_ground_truth_masks = {img_id: masks.get(class_id, []) for img_id, masks in ground_truth_masks.items()}

        # Count total number of ground truth instances for this class
        total_ground_truth = sum(len(masks) for masks in class_ground_truth_masks.values())

        if total_ground_truth == 0 and not class_predictions:
             class_aps[class_id] = 0.0 # AP is 0 if no GT or predictions
             continue
        elif total_ground_truth == 0 and class_predictions:
             class_aps[class_id] = 0.0 # All predictions are FP if no GT
             continue
        elif total_ground_truth > 0 and not class_predictions:
             class_aps[class_id] = 0.0 # All GT are FN if no predictions
             continue


        # Lists to store true positives and false positives at each prediction
        tps = []
        fps = []

        # Keep track of matched ground truth masks
        matched_gt_masks_for_class = {img_id: [False] * len(masks) for img_id, masks in class_ground_truth_masks.items()}


        for pred in class_predictions:
            image_id = pred['image_id']
            pred_mask = pred['mask']
            gt_masks_for_image = class_ground_truth_masks.get(image_id, [])

            best_iou = 0
            best_gt_match_index = -1

            # Check if there are any GT masks for this class in this image before iterating
            if gt_masks_for_image:
                # Get the matched status for this image and class
                matched_flags_for_class = matched_gt_masks_for_class.get(image_id, [])


                for i, gt_mask in enumerate(gt_masks_for_image):
                     # Only consider ground truth masks that haven't been matched yet
                    if i < len(matched_flags_for_class) and not matched_flags_for_class[i]:
                        # Ensure predicted and ground truth masks have the same shape
                        if pred_mask.shape != gt_mask.shape:
                            iou = 0.0 # Treat as no overlap if shapes don't match
                        else:
                            iou = calculate_iou_segmentation(pred_mask, gt_mask)

                        if iou > best_iou:
                            best_iou = iou
                            best_gt_match_index = i

            if best_iou >= iou_threshold:
                tps.append(1)
                fps.append(0)
                # Mark the matched ground truth if a match was found
                if best_gt_match_index != -1:
                    matched_gt_masks_for_class[image_id][best_gt_match_index] = True
            else:
                tps.append(0)
                fps.append(1)


        ap = calculate_ap(tps, fps, total_ground_truth)
        class_aps[class_id] = ap

    return class_aps # Return dictionary of class APs


def bootstrap_mAP_ci(all_image_predictions, ground_truth_masks, num_bootstrap_samples=1000, confidence=0.95, iou_threshold=0.5, class_names=None):
    """
    Performs bootstrapping to calculate confidence intervals for class-wise and overall mAP.
    all_image_predictions is a list of all prediction dictionaries across all images with 'image_id'.
    ground_truth_masks is the dictionary of all ground truth masks {image_id: {class_id: [mask]}}.
    """
    # Use image IDs present in ground truth for resampling as this defines the dataset extent
    image_ids = sorted(list(ground_truth_masks.keys()))
    n_images = len(image_ids)

    # Determine all unique class IDs from both ground truth and predictions
    predicted_class_ids = list(set([p['class_id'] for p in all_image_predictions]))
    gt_class_ids = set()
    for image_id, class_masks in ground_truth_masks.items():
        gt_class_ids.update(class_masks.keys())
    unique_class_ids = sorted(list(set(predicted_class_ids + list(gt_class_ids))))


    bootstrap_class_aps = defaultdict(list) # {class_id: [ap_values]}
    bootstrap_overall_maps = [] # [overall_map_values]

    # Group all predictions by image ID for easier resampling
    all_predictions_by_image = defaultdict(list)
    for pred in all_image_predictions:
        all_predictions_by_image[pred['image_id']].append(pred)


    print(f"Performing bootstrapping for mAP50 with {num_bootstrap_samples} samples...")

    for _ in range(num_bootstrap_samples):
        # Resample image IDs with replacement
        resampled_image_ids = resample(image_ids, n_samples=n_images, replace=True)

        # Collect predictions and ground truth masks for the resampled images
        resampled_predictions_flat = []
        resampled_ground_truth_masks = {}

        for img_id in resampled_image_ids:
            # Collect predictions for this image
            resampled_predictions_flat.extend(all_predictions_by_image.get(img_id, []))
            # Collect ground truth for this image
            if img_id in ground_truth_masks:
                resampled_ground_truth_masks[img_id] = ground_truth_masks[img_id]


        # Calculate AP for each class and overall mAP for the resampled data
        resampled_class_aps = calculate_mAP_segmentation_for_bootstrap(
            resampled_predictions_flat,
            resampled_ground_truth_masks,
            iou_threshold=iou_threshold
        )

        # Store class APs
        for class_id in unique_class_ids:
             # Append the calculated AP for this class, or 0 if the class was not present in the resample GT
            bootstrap_class_aps[class_id].append(resampled_class_aps.get(class_id, 0.0))


        # Calculate overall mAP by averaging class APs for this sample
        # Only average classes that were present in the resampled ground truth for a meaningful average
        # Or average over all unique classes if present in either GT or Pred in the resample
        # Let's average over all unique classes from the *full* dataset if they had *any* detections or GT in the resample
        present_class_ids_in_resample = set(resampled_class_aps.keys())

        # Filter unique_class_ids to only those present in the resample
        mappable_class_aps = [ap for class_id, ap in resampled_class_aps.items() if class_id in unique_class_ids]


        overall_map = np.mean(mappable_class_aps) if mappable_class_aps else 0.0 # Average only non-zero APs if no classes had matches
        bootstrap_overall_maps.append(overall_map)


    # Calculate mean and confidence intervals from bootstrap results
    metric_summary = {}
    alpha = 1.0 - confidence
    class_names = model.names # Get class names from the model

    # Calculate CI for each class AP
    for class_id in unique_class_ids:
        class_name = class_names.get(class_id, f'Class {class_id}')
        ap_values = np.array(bootstrap_class_aps[class_id])

        if ap_values.size > 0 and not np.all(ap_values == 0): # Calculate CI only if there are non-zero AP values
            mean_ap = np.mean(ap_values)
            lower_bound = np.percentile(ap_values, alpha / 2.0 * 100)
            upper_bound = np.percentile(ap_values, (1.0 - alpha / 2.0) * 100)
            metric_summary[class_name] = {
                'Mean': mean_ap,
                'CI_Lower': lower_bound,
                'CI_Upper': upper_bound
            }
        else:
             # If all AP values are 0 or no values (class not in resamples), CI is 0,0
             metric_summary[class_name] = {'Mean': 0.0, 'CI_Lower': 0.0, 'CI_Upper': 0.0}


    # Calculate CI for overall mAP
    if bootstrap_overall_maps:
        mean_overall_map = np.mean(bootstrap_overall_maps)
        lower_bound_overall = np.percentile(bootstrap_overall_maps, alpha / 2.0 * 100)
        upper_bound_overall = np.percentile(bootstrap_overall_maps, (1.0 - alpha / 2.0) * 100)
        metric_summary['Overall'] = {
            'Mean': mean_overall_map,
            'CI_Lower': lower_bound_overall,
            'CI_Upper': upper_bound_overall
        }
    else:
        metric_summary['Overall'] = {'Mean': 0.0, 'CI_Lower': 0.0, 'CI_Upper': 0.0}


    return metric_summary


# Assuming 'results' from model.predict() is available and 'ground_truth_segmentation_masks' is loaded

# Example of how to process results from model.predict() for segmentation
# The 'results' object from model.predict() is a list of Results objects
predictions = {}
# Add checks for the structure of 'results'
if isinstance(results, list) and all(hasattr(r, 'masks') for r in results):
    for result in results:
        image_path = result.path
        image_id = os.path.splitext(os.path.basename(image_path))[0]
        predictions[image_id] = []
        if result.masks is not None:
            # result.masks is a Masks object
            # result.masks.data contains the boolean masks as a tensor (N, H, W)
            # result.boxes contains bounding box info, including class_id and confidence
            if hasattr(result.masks, 'data') and hasattr(result.boxes, 'cls') and hasattr(result.boxes, 'conf'):
                masks_tensor = result.masks.data # Use .data for the tensor
                class_ids = result.boxes.cls
                scores = result.boxes.conf

                # Predicted masks are likely already at the processing size (e.g., 448x768)
                # We will use their shape as the target size for resizing GT masks.
                # Assuming all predicted masks for an image have the same shape.
                if masks_tensor.shape[0] > 0:
                    predicted_mask_shape = masks_tensor.shape[1:] # (H, W)
                    target_size = (predicted_mask_shape[1], predicted_mask_shape[0]) # (W, H)
                else:
                    # Fallback if no masks are predicted for this image
                    target_size = (768, 448) # Default target size, should match expected prediction size


                for i in range(masks_tensor.shape[0]):
                     # Ensure mask tensor is on CPU before converting to numpy
                     pred_mask_np = masks_tensor[i].cpu().numpy().astype(bool)

                     predictions[image_id].append({
                        'class_id': int(class_ids[i]),
                        'mask': pred_mask_np, # Use the predicted mask as is
                        'score': float(scores[i])
                    })
            else:
                 print(f"Warning: Unexpected mask or box format in result for image {image_id}")
else:
    print("Error: 'results' variable is not in the expected format for processing segmentation predictions.")
    predictions = {} # Reset predictions to empty if format is wrong


# Determine the target size for resizing ground truth masks based on predicted mask shape
target_size_for_gt = (768, 448) # Default, will try to get from predictions if available
if predictions:
    # Find the shape of the first predicted mask in the first image with predictions
    for image_id, predicted_annotations in predictions.items():
        if predicted_annotations and predicted_annotations[0].get('mask') is not None:
            predicted_mask_shape = predicted_annotations[0]['mask'].shape # (H, W)
            target_size_for_gt = (predicted_mask_shape[1], predicted_mask_shape[0]) # (W, H)
            print(f"Determined target size for GT masks from predictions: {target_size_for_gt}")
            break # Found a predicted mask shape, no need to check further

# Load ground truth segmentation masks, resizing them to match predicted mask size
labels_folder = '/content/dataset/YOLO_Test_266/labels/train'
image_folder = '/content/dataset/YOLO_Test_266/images/train' # Need image folder to get image sizes
ground_truth_segmentation_masks = load_ground_truth_segmentation_masks(labels_folder, image_folder, target_size=target_size_for_gt)


# Prepare flattened list of all predictions with image_id for mAP50 bootstrapping
all_image_predictions_flat = []
for image_id, predicted_annotations in predictions.items():
    for pred in predicted_annotations:
         all_image_predictions_flat.append({
            'image_id': image_id,
            'class_id': pred['class_id'],
            'mask': pred['mask'],
            'score': pred['score']
        })


# Calculate class-wise and overall mAP50 with 95% Confidence Interval using bootstrapping
map50_ci_summary = bootstrap_mAP_ci(
    all_image_predictions_flat,
    ground_truth_segmentation_masks,
    num_bootstrap_samples=1000, # Number of bootstrap samples
    confidence=0.95,
    iou_threshold=0.5,
    class_names=model.names
)

print("\nmAP50 with 95% Confidence Interval (Segmentation):")

# Convert to DataFrame for display
map_ci_df = pd.DataFrame(map50_ci_summary).T
map_ci_df = map_ci_df.reindex(['Mean', 'CI_Lower', 'CI_Upper'], axis=1) # Ensure column order


display(map_ci_df)

# You can also save this DataFrame to a file if needed
 map_ci_df.to_csv('/content/map50_segmentation_with_ci.csv')
# print("\nmAP50 with 95% CI saved to /content/map50_segmentation_with_ci.csv")

In [ ]:
# Combine all calculated metrics into a single comprehensive table

import pandas as pd

# Assuming the following variables are available from previous cell executions:
# metric_summary_with_ci (from dc3951e2): PRF, Accuracy, Specificity with CI
# dice_iou_ci_summary (from a6b82e6d): Class-wise Mean Dice/IoU with CI, Overall Pixel Dice/IoU with CI
# map50_ci_summary (from fccb4962): Class-wise and Overall mAP50 with CI


# --- Restructure and combine data from metric_summary_with_ci (PRF, Accuracy, Specificity) ---
combined_data_for_df = {}
if 'metric_summary_with_ci' in locals() and metric_summary_with_ci is not None:
    for class_name, metrics in metric_summary_with_ci.items():
        if class_name not in combined_data_for_df:
            combined_data_for_df[class_name] = {}
        for metric_name, values in metrics.items():
            combined_data_for_df[class_name][(metric_name, 'Mean')] = values.get('Mean', np.nan)
            combined_data_for_df[class_name][(metric_name, 'CI_Lower')] = values.get('CI_Lower', np.nan)
            combined_data_for_df[class_name][(metric_name, 'CI_Upper')] = values.get('CI_Upper', np.nan)
else:
    print("Warning: metric_summary_with_ci not found or is None. Skipping PRF, Accuracy, Specificity inclusion.")


# --- Add data from dice_iou_ci_summary (Dice, IoU) ---
if 'dice_iou_ci_summary' in locals() and dice_iou_ci_summary is not None:
    for class_name, metrics in dice_iou_ci_summary.items():
         if class_name not in combined_data_for_df:
            combined_data_for_df[class_name] = {}

         for metric_name, values in metrics.items():
             combined_data_for_df[class_name][(metric_name, 'Mean')] = values.get('Mean', np.nan)
             combined_data_for_df[class_name][(metric_name, 'CI_Lower')] = values.get('CI_Lower', np.nan)
             combined_data_for_df[class_name][(metric_name, 'CI_Upper')] = values.get('CI_Upper', np.nan)
else:
    print("Warning: dice_iou_ci_summary not found or is None. Skipping Dice/IoU inclusion.")


# --- Add data from map50_ci_summary (mAP50) ---
if 'map50_ci_summary' in locals() and map50_ci_summary is not None:
    for class_name, values in map50_ci_summary.items():
        if class_name not in combined_data_for_df:
            combined_data_for_df[class_name] = {}

        # mAP50 metric name is just 'mAP50'
        combined_data_for_df[class_name][('mAP50', 'Mean')] = values.get('Mean', np.nan)
        combined_data_for_df[class_name][('mAP50', 'CI_Lower')] = values.get('CI_Lower', np.nan)
        combined_data_for_df[class_name][('mAP50', 'CI_Upper')] = values.get('CI_Upper', np.nan)
else:
    print("Warning: map50_ci_summary not found or is None. Skipping mAP50 inclusion.")


# --- Create the DataFrame ---
# Use sorted class names for consistent row order, ensuring 'Overall' is last
class_order = sorted(combined_data_for_df.keys())
if 'Overall' in class_order:
    class_order.remove('Overall')
    class_order.append('Overall')


# Create the DataFrame directly from the restructured dictionary, ensuring row order
combined_metrics_df = pd.DataFrame.from_dict(combined_data_for_df, orient='index')
combined_metrics_df = combined_metrics_df.reindex(class_order)


# --- Create and sort the MultiIndex columns ---
# Get all unique column tuples (Metric Name, Detail) across all classes
all_column_tuples = sorted(list(set([col for cols in combined_data_for_df.values() for col in cols.keys()])))

# Ensure the column index is a MultiIndex
if not isinstance(combined_metrics_df.columns, pd.MultiIndex):
     combined_metrics_df.columns = pd.MultiIndex.from_tuples(combined_metrics_df.columns)


# Optional: Define a preferred order for the main metric categories
preferred_metric_order = [
    'Precision', 'Recall', 'F1-score',
    'Accuracy', 'Specificity',
    'Mean Dice', 'Mean IoU',
    'Overall Dice', 'Overall IoU',
    'mAP50'
]

# Create a mapping for sorting based on the preferred order
# Use a default high value for metrics not in the preferred list
col_order_mapping = {col_name: preferred_metric_order.index(col_name) if col_name in preferred_metric_order else len(preferred_metric_order) for col_name in combined_metrics_df.columns.get_level_values(0).unique()}

# Sort the columns based on the order of the first level using the mapping
sorted_col_tuples = sorted(combined_metrics_df.columns, key=lambda x: (col_order_mapping.get(x[0], len(preferred_metric_order)), x[1])) # Sort by metric name then detail ('Mean', 'CI_Lower', 'CI_Upper')

# Reindex the DataFrame with the sorted column order
combined_metrics_df = combined_metrics_df[sorted_col_tuples]


# --- Display the final table ---
print("\nComprehensive Performance Metrics with 95% Confidence Intervals (Segmentation):")
display(combined_metrics_df)


# You can also save this DataFrame to a file if needed
# combined_metrics_df.to_csv('/content/comprehensive_metrics_final.csv')
# print("\nFinal comprehensive metrics with 95% CI saved to /content/comprehensive_metrics_final.csv")

In [ ]:
import os
import shutil

# Define source file and destination directory
source_file = '/content/mask_custom.tar'
destination_dir = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Test_compare'

print(f"Checking if source file exists: {source_file}")
# Check if the source file exists
if not os.path.exists(source_file):
    print(f"Error: Source file not found: {source_file}")
else:
    print(f"Source file found. Checking if destination directory exists: {destination_dir}")
    # Create the destination directory if it doesn't exist
    if not os.path.exists(destination_dir):
        print(f"Destination directory does not exist. Creating: {destination_dir}")
        try:
            os.makedirs(destination_dir)
            print(f"Successfully created directory: {destination_dir}")
        except OSError as e:
            print(f"Error creating directory {destination_dir}: {e}")
            # Stop execution if directory cannot be created
            raise

    # Define the full destination path for the file
    destination_path = os.path.join(destination_dir, os.path.basename(source_file))

    print(f"Copying {source_file} to {destination_path}...")
    try:
        shutil.copy2(source_file, destination_path) # copy2 attempts to preserve metadata
        print(f"Successfully copied: {source_file} to {destination_dir}")
    except IOError as e:
        print(f"Error copying file {source_file}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred while copying {source_file}: {e}")

print("Finished attempting to copy the tar file.")

In [ ]:
import os

output_folder = '/content/runs/mask_Yolo'

print(f"Checking contents of folder: {output_folder}")
if os.path.exists(output_folder):
    files_in_folder = os.listdir(output_folder)
    if files_in_folder:
        print(f"Found {len(files_in_folder)} files in {output_folder}:")
        # Print the first few files to give an idea of the content
        for i, file_name in enumerate(files_in_folder[:10]):
            print(file_name)
        if len(files_in_folder) > 10:
            print("...")
    else:
        print(f"The folder {output_folder} exists but is empty.")
else:
    print(f"The folder {output_folder} does not exist.")

In [ ]:
import os
import pandas as pd
from PIL import Image
import numpy as np
from collections import defaultdict

# Define the folder where combined per-class masks will be saved
masks_folder = '/content/runs/mask_Yolo_combined_per_class' # Updated folder path

# Dictionary to store calculated areas per image and class
image_areas = defaultdict(lambda: {'Tooth': 0, 'Root': 0})

# Get a list of all saved mask files
mask_files = [f for f in os.listdir(masks_folder) if f.endswith('.png')]

if not mask_files:
    print(f"No mask files found in {masks_folder}.")
else:
    print(f"Found {len(mask_files)} mask files in {masks_folder}. Calculating areas...")

    # Process each mask file
    for mask_file in mask_files:
        # Assuming filename format is image_id_combined_mask_ClassName.png
        parts = mask_file.split('_combined_mask_') # Split by "_combined_mask_"

        if len(parts) == 2:
            image_id = parts[0] # Part before "_combined_mask_" is the image_id
            class_name_part = parts[1] # Part after "_combined_mask_"

            # Extract class name (remove .png extension)
            class_name = os.path.splitext(class_name_part)[0]

            # Ensure extracted class_name is one of the expected classes
            if class_name in ['Tooth', 'Root']:
                mask_path = os.path.join(masks_folder, mask_file)
                try:
                    # Open the mask image and convert to numpy array
                    mask_img = Image.open(mask_path).convert('L') # Convert to grayscale
                    mask_np = np.array(mask_img)

                    # Assuming white pixels (value 255) represent the mask area
                    # Count the number of white pixels (mask area)
                    area = np.sum(mask_np > 0) # Count pixels with value > 0 (assuming 0 or 255)

                    # Add the area to the corresponding image_id and class_name
                    # Since these are combined per-class masks, we just assign the area
                    image_areas[image_id][class_name] = area


                except Exception as e:
                    print(f"Error processing mask file {mask_file}: {e}. Skipping.")

            else:
                 print(f"Warning: Extracted unexpected class name '{class_name}' from file {mask_file}. Skipping area calculation for this mask.")

        else:
             print(f"Warning: Skipping mask file with unexpected name format (missing '_combined_mask_'): {mask_file}")


    # Convert the collected areas into a pandas DataFrame
    if image_areas:
        # Build the DataFrame from the collected image_areas dictionary
        areas_df = pd.DataFrame.from_dict(image_areas, orient='index')

        # Ensure 'Tooth' and 'Root' columns exist, adding them with 0 if not
        # This handles cases where a class mask wasn't found for an image
        if 'Tooth' not in areas_df.columns:
            areas_df['Tooth'] = 0
        if 'Root' not in areas_df.columns:
            areas_df['Root'] = 0

        # Reorder columns for consistent output
        areas_df = areas_df[['Tooth', 'Root']]

        # Sort by image_id (index) for consistent order
        areas_df = areas_df.sort_index()

        # Save the DataFrame to a CSV file
        output_csv_path = '/content/yolo_mask_areas.csv' # Can save to a temporary path first
        areas_df.to_csv(output_csv_path)

        print(f"\nArea calculation complete. Results saved to {output_csv_path}")
        display(areas_df.head()) # Display the head of the DataFrame
    else:
        print("\nNo image areas were calculated.")

In [ ]:
import os
import pandas as pd

# Assuming 'areas_df' DataFrame is available from previous steps

# Define the destination path in Google Drive
destination_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/Result_compare_0729_best/yolo_mask_areas.csv'

# Define the directory part of the destination path
destination_dir = os.path.dirname(destination_path)

print(f"Checking if destination directory exists: {destination_dir}")
# Create the destination directory if it doesn't exist
if not os.path.exists(destination_dir):
    print(f"Destination directory does not exist. Creating: {destination_dir}")
    try:
        os.makedirs(destination_dir)
        print(f"Successfully created directory: {destination_dir}")
    except OSError as e:
        print(f"Error creating directory {destination_dir}: {e}")
        # Stop execution if directory cannot be created
        # raise # Uncomment this line if you want to stop execution on error

# Check if areas_df exists before saving
if 'areas_df' in locals() and isinstance(areas_df, pd.DataFrame):
    print(f"Saving areas_df to {destination_path}")
    try:
        areas_df.to_csv(destination_path)
        print("Successfully saved areas_df to Google Drive.")
    except Exception as e:
        print(f"Error saving areas_df to Google Drive: {e}")
else:
    print("Error: 'areas_df' DataFrame not found. Cannot save the file.")

## Train fold_4 결과 확인-- 원본이미지와 mask 이미지 같이 표시

In [ ]:
import tarfile

dataset_tar_path = '/content/drive/MyDrive/Colab Notebooks/Root_seg/YOLO_0725_Result_best/fold4_backup.tar.gz'
data_path = '/content'

print(f"Extracting {dataset_tar_path} to {data_path}...")
with tarfile.open(dataset_tar_path, mode="r:gz") as tar:  # r:gz = gzip 압축 파일 자동 인식
    tar.extractall(path=data_path)

print("Extraction complete.")